In [ ]:
#%pip install pip --upgrade
#%pip install --upgrade pip setuptools wheel

In [ ]:
#%pip install langchain openai neo4j langchain-openai textdistance pandas spacy\
#    langchain-community seaborn openpyxl chromadb markdown bs4 pypdf neo4j-graphrag\
#        https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

#!/usr/local/bin/python3.13 -m spacy download en_core_web_md
#!python3 -m spacy download en_core_web_md

In [ ]:
import os
import shutil
import json
import re
import pandas as pd
import tiktoken
import math
import spacy
from typing import Set, Any, Union, Dict, List, Tuple, Hashable
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import chromadb
import textdistance
from tqdm.auto import tqdm
from neo4j.exceptions import Neo4jError, ClientError

# On MAC:
!export HNSWLIB_NO_NATIVE=1

tqdm.pandas()

In [ ]:
# Set model name
model_name="gpt-4o" # "gpt-4o-mini", IDs of OpenAI's FT models "ft:gpt-4o-2024-08-06:..."

DATA_PATH = "./NLquestions"
CYPHER_PATH = "./CypherQueries"
NEO4J_OUTPUTS_PATH = "./Neo4jOutputs"

os.environ["NEO4J_URI"] = "neo4j+s://helix.biodata.di.unimi.it:7687"
os.environ["NEO4J_USERNAME"] = "Text2Cypher"
os.environ["NEO4J_PASSWORD"] = "Text2Cypher"
os.environ["NEO4J_DATABASE"] = "mirnakgt2c"

graph = Neo4jGraph()
graph.refresh_schema()
print(graph.schema)

***
# Creating the dataset

In [ ]:
import json
import pandas as pd

def load_level(path, level_name):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data).reset_index(drop=True)
    df["ID"] = f"{level_name}/question_" + (df.index + 1).astype(str)
    df["level"] = level_name
    return df

df_nl   = load_level("FTdataset/nodeLevel.json", "nodeLevel")
df_1hop = load_level("FTdataset/1hop.json",      "1hop")
df_2hop = load_level("FTdataset/2hop.json",      "2hop")
df_3hop = load_level("FTdataset/3hop.json",      "3hop")
df_hl = load_level("FTdataset/hardLevel.json",      "hardLevel")

df_all = pd.concat([df_nl, df_1hop, df_2hop, df_3hop, df_hl], ignore_index=True)
# The following is useful to check the length of the longest cypher query and limit the max number of tokens accordingly (this speed ups evaluation)
print("Longest cypher:", df_all["cypher"].str.len().max())
longest = df_all[df_all["cypher"].str.len() == df_all["cypher"].str.len().max()]['cypher']
print(longest)
encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(longest.iloc[0])
MAX_TOKENS = math.ceil(len(tokens) / 100) * 100
print(len(tokens), MAX_TOKENS)

test_df  = df_all.groupby("level", group_keys=False).sample(frac=0.10, random_state=42)
train_df = df_all.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df[['question', 'cypher']].to_json("FTdataset_local.json", orient="records", indent=2)

SYSTEM_PROMPT = (
    "Given an input question, convert it to a Cypher query. No pre-amble."
)

def row_to_chat_example(row, add_system=True):
    messages = []
    if add_system:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": row["question"]})
    messages.append({"role": "assistant", "content": row["cypher"]})
    return {"messages": messages}

out_path = "FTdataset_GPTviaGUI.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        ex = row_to_chat_example(row, add_system=True)  
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_df.head(n=3)

In [ ]:
test_df.head(n=3)

In [ ]:
test_df["cypher"].iloc[244]

In [ ]:
test_df = load_level("50-biologists.json", "biologists")
test_df

***
# Pre-processing for building a vector store for performing RAG
## Step-by-step tutorial for Creating a Vector Store with ChromaDB

In [ ]:
def build_skip_set(test_df):
    skip = set()
    for qid in test_df["ID"]:
        skip.add(f"{qid}.txt")
    return skip

skip_files = build_skip_set(test_df)

def load_txt_documents():
    """
    Load .txt documents from the specified directory.
    Returns: List of Document objects
    """
    all_documents = []

    for folder_name in os.listdir(DATA_PATH):
        folder_path = os.path.join(DATA_PATH, folder_name)
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                if file_name.endswith(".txt") and folder_name + "/" + file_name not in skip_files:
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text_content = f.read()
                            all_documents.append(Document(
                                page_content=text_content,
                                metadata={"type": "text", "source": folder_name + "/" + file_name}
                            ))
                    except Exception as e:
                        print(f"Error loading TXT file {file_name}: {e}")

    return all_documents

Check if it works.

In [ ]:
documents = load_txt_documents()

bad = [
    d.metadata["source"]
    for d in documents
    if os.path.basename(d.metadata["source"]) in skip_files
]

assert len(bad) == 0, f"LEAKAGE! Indexed test files: {bad[:5]}"

In [ ]:
documents = load_txt_documents()

documents[1].page_content, documents[1].metadata['source']

***

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = "...YOUR_API_KEY_HERE..."  # Replace with your OpenAI API key
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
chroma_client = chromadb.PersistentClient(path="./chroma_db")

⚠️ Uncomment and run the following chunk only if you want to recreate the collection from scratch!

In [ ]:
'''# Re-create collection if it previously existed
try:
    chroma_client.delete_collection("my_collection")
except:
    pass

collection = chroma_client.create_collection(name="my_collection")
texts = [doc.page_content for doc in documents]
ids = [doc.metadata["source"] for doc in documents]
metas = [doc.metadata for doc in documents]

BATCH = 512

for start in range(0, len(texts), BATCH):
    batch_texts = texts[start:start+BATCH]
    batch_ids   = ids[start:start+BATCH]
    batch_metas = metas[start:start+BATCH]

    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=batch_texts
    )
    batch_embeddings = [item.embedding for item in resp.data]

    collection.add(
        documents=batch_texts,
        ids=batch_ids,
        metadatas=batch_metas,
        embeddings=batch_embeddings
    )'''

In [ ]:
# Check if it works
collection = chroma_client.get_collection(name="my_collection")
query = "Find all gene names containing 'MIR'"

query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Match {i+1}:")
    print("→", doc)
    print("Filename:", results["metadatas"][0][i]["source"])
    print("Distance:", results["distances"][0][i])
    print("-" * 40)

***
# Metrics

We will use the following metrics to evaluate the Cypher generation ability of LLMs.
* **Jaro-Winkler**
* **Jaccard**
* **Coverage**
* **Pass**:

In [ ]:
def get_jw_distance(string1: str, string2: str) -> float:
    """
    Calculate the Jaro-Winkler distance between two strings.

    The Jaro-Winkler distance is a measure of similarity between two strings.
    The score is normalized such that 0 equates to no similarity and
    1 is an exact match.
    """
    # Call the jaro_winkler function from the textdistance library.
    return textdistance.jaro_winkler(string1, string2)

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple

from typing import Any, Hashable

from collections.abc import Mapping, Iterable
from typing import Any, Hashable

def _norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    if isinstance(v, Mapping):
        return frozenset(_norm_value(x) for x in v.values())

    if isinstance(v, (list, tuple)):
        return frozenset(_norm_value(x) for x in v)

    if isinstance(v, set):
        return frozenset(_norm_value(x) for x in v)

    return v if isinstance(v, Hashable) else str(v)


def rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[Set], List[Set]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [{_norm_value(v) for v in row.values()} for row in dictL]
    setViewsR = [{_norm_value(v) for v in row.values()} for row in dictR]
    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = rowsim(setViewsL[i], setViewsR[j])
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL
    return setViewsL, setViewsR

def df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        view_L = [[_norm_value(v) for v in row.values()] for row in dictL]
        view_R = [[_norm_value(v) for v in row.values()] for row in dictR]
    else:
        view_L, view_R = make_alignment(dictL, dictR)

    totalL = {(i, x) for i, row in enumerate(view_L) for x in set(row)}
    totalR = {(i, x) for i, row in enumerate(view_R) for x in set(row)}

    union = totalL | totalR
    return 1.0 if not union else len(totalL & totalR) / len(union)

def jaccard_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    return df_sim(dict_L, dict_R, "order by" in f"{cypher_L}".lower())

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple
from collections.abc import Mapping

def t2c_norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    return v if isinstance(v, Hashable) else str(v)

def t2c_flat_values(v: Any) -> Set[Hashable]:
    vals: Set[Hashable] = set()

    if isinstance(v, (str, int, float)):
        vals.add(t2c_norm_value(v))
        return vals

    if isinstance(v, Mapping):
        for x in v.values():
            vals |= t2c_flat_values(x)
        return vals

    if isinstance(v, (list, tuple, set)):
        for x in v:
            vals |= t2c_flat_values(x)
        return vals

    vals.add(t2c_norm_value(v))
    return vals


def t2c_row_values(row: Dict) -> Set[Hashable]:
    s: Set[Hashable] = set()
    for v in row.values():
        s |= t2c_flat_values(v)
    return s

def t2c_rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def t2c_make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[List[Hashable]], List[List[Hashable]]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [list(t2c_row_values(row)) for row in dictL]
    setViewsR = [list(t2c_row_values(row)) for row in dictR]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = t2c_rowsim(set(setViewsL[i]), set(setViewsR[j]))
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    return setViewsL, setViewsR

def t2c_row_attr_recall(gt_row: Dict, pred_values: Set[Hashable]) -> float:
    num_attrs = len(gt_row)
    if num_attrs == 0:
        return 1.0

    matched = 0
    for v in gt_row.values():
        gt_vals = t2c_flat_values(v)
        if gt_vals & pred_values:
            matched += 1

    return matched / num_attrs

def t2c_df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        lenL, lenR = len(dictL), len(dictR)
        N = max(lenL, lenR)

        if N == 0:
            return 1.0 

        row_scores: List[float] = []

        for i in range(N):
            if i < lenL:
                gt_row = dictL[i]
                pred_vals = t2c_row_values(dictR[i]) if i < lenR else set()
                row_scores.append(t2c_row_attr_recall(gt_row, pred_vals))
            else:
                pred_vals = t2c_row_values(dictR[i])
                row_scores.append(0.0 if pred_vals else 1.0)

        return sum(row_scores) / N

    view_L, view_R = t2c_make_alignment(dictL, dictR)

    total_attrs = 0
    matched_attrs = 0

    for i, gt_row in enumerate(dictL):
        total_attrs += len(gt_row)
        pred_vals = set(view_R[i]) if i < len(view_R) else set()
        row_recall = t2c_row_attr_recall(gt_row, pred_vals)
        matched_attrs += row_recall * len(gt_row)

    if total_attrs == 0:
        return 1.0

    return matched_attrs / total_attrs

def coverage_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    order_sensitive = "order by" in f"{cypher_L}".lower()
    return max(t2c_df_sim(dict_L, dict_R, order_sensitive), df_sim(dict_L, dict_R, order_sensitive))


***
## Generating Cypher statements

In [ ]:
llm = ChatOpenAI(model_name=model_name, temperature=0)
model_name = "gpt-4o" # "gpt-4o-mini", "FTgpt-4o", , "FTgpt-4o-mini"

***
# Examples via RAG

Examples reteieved from a vector store in a RAG fashion are useful to provide context to the LLM. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NLQUESTIONS_PATH = Path("./NLquestions")   
CYPHER_PATH      = Path("./CypherQueries")  

# -----------------------------
# Helpers: mapping source -> cypher file
# -----------------------------
def cypher_file_from_source(meta_source: str) -> Path:
    """
    meta_source: es '3hop/question_3.txt'
    -> CypherQueries/3hop/question_3.txt
    """
    return CYPHER_PATH / Path(meta_source)


# -----------------------------
# RAG: retrieve examples bundle
# -----------------------------
def retrieve_examples_bundle(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source.replace("question", "cypher"))
        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()
            
        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }


def enrich_with_examples(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle(inputs["question"], n_results=3)
    return {
        **inputs,
        "examples": bundle["examples_text"],                
        "example_ids": bundle["example_ids"],                
        "example_distances": bundle["example_distances"],    
    }

In [ ]:
import re

def _normalize_generated_cypher(cypher) -> str:
    if cypher is None:
        return ""

    if not isinstance(cypher, str):
        cypher = str(cypher)

    cypher = cypher.strip()
    cypher = re.sub(r"^```(?:cypher)?\s*", "", cypher, flags=re.IGNORECASE)
    cypher = re.sub(r"\s*```$", "", cypher)

    cypher = cypher.replace("\r\n", "\n").replace("\r", "\n").strip()

    cypher = cypher.replace("\x08", r"\b")

    cypher = re.sub(r'(?<!\\)\\b', r'\\\\b', cypher)

    return cypher

In [ ]:
cypher_template = """Based on the examples below that provide useful
patterns for querying the graph database,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Examples:
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given an input question, convert it to a Cypher query. No pre-amble."),
    ("human", cypher_template),
])

cypher_generate = (
    cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)


def _run_graph_query_safe(graph, cypher: str):
    try:
        return graph.query(cypher)
    except ClientError as e:
        if e.code == "Neo.ClientError.Statement.SyntaxError":
            return [{"id": "Cypher syntax error"}]
        if e.code == "Neo.ClientError.Statement.SemanticError":
            return [{"id": "Cypher semantic error"}]
        return [{"id": f"Cypher client error ({e.code})"}]
    except Neo4jError:
        return [{"id": "Neo4j runtime error"}]
    except Exception:
        return [{"id": "Other error"}]


def evaluate_cypher_queries_RAG(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, k: int = 1) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):

        true_data = _run_graph_query_safe(graph, _normalize_generated_cypher(row["cypher"]))

        example_generated_cyphers = []
        example_eval_datas = []
        example_ids_runs = []
        example_dist_runs = []
        prompts = []

        for _ in range(k):
            enriched = enrich_with_examples({"question": row["question"]})

            example_ids_runs.append(enriched["example_ids"])
            example_dist_runs.append(enriched["example_distances"])

            formatted_prompt = cypher_prompt.format_messages(
                question=enriched["question"],
                examples=enriched["examples"]
            )
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })

            cypher = cypher_generate.invoke({
                "question": enriched["question"],
                "examples": enriched["examples"]
            })

            example_generated_cyphers.append(cypher)
            example_eval_datas.append(_run_graph_query_safe(graph, _normalize_generated_cypher(cypher)))

        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        passs = 1 if coverage == 1 else 0

        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        prompt.append(prompts)

        retrieved_example_ids.append(example_ids_runs)
        retrieved_example_distances.append(example_dist_runs)

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverages
    out["pass"] = passes
    out["prompt"] = prompt

    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

In [ ]:
inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")

In [ ]:
df = evaluate_cypher_queries_RAG(test_df, graph, cypher_generate, cypher_prompt)
df.head(3)

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG_biologists.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG_biologists.pkl")
df.head(n=3)